# TrafficSense - Day 1: Data Pipeline
**PS2 Hackathon Build** | Bengaluru Incident Congestion Intelligence

Loads the raw ASTRAM event export, cleans it, engineers features, and saves a pruned, model-ready dataset. Key fixes over the original plan: EDA-derived peak hours (not standard India rush hours), strictly past-only incident density (no leakage), haversine-only HDBSCAN clustering, targeted Kannada NLP translation (638 records, not all 8K), real osmnx centrality instead of a faked proxy, and a correlation/MI pass that prunes ~36 engineered columns down to the ~14 that actually carry signal.

## Setup
Centralized paths so every step writes to the same `data/processed`, `plots`, `models/trained` locations.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import time
import warnings
import joblib
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
%matplotlib inline

RAW_CSV = 'data/raw/astram_events.csv'
PROCESSED_DIR = 'data/processed'
PLOTS_DIR = 'plots'
MODELS_DIR = 'models/trained'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print('Setup complete.')

## 1. Load & Clean
Load the raw CSV, parse datetimes into `resolution_min` (capped at p99, with negative/stale-active rows nulled out), and clean the categorical columns: `corridor`/`zone` nulls backfilled, `event_cause` standardized and rare planned-event types merged, vehicle type/road-closure/priority normalized, GPS validated, duplicates flagged.

In [ ]:
df = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nNull counts (top 15):')
df.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
for col in ['event_type', 'event_cause', 'status', 'priority', 'requires_road_closure']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False).head(10))

print(f'\nCorridor null/NULL rate: {(df["corridor"].isna() | (df["corridor"] == "NULL")).sum()} / {len(df)}')
print(f'Zone null/NULL rate: {(df["zone"].isna() | (df["zone"] == "NULL")).sum()} / {len(df)}')

In [ ]:
for col in ['start_datetime', 'resolved_datetime', 'closed_datetime', 'modified_datetime', 'created_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

df['end_time'] = df['resolved_datetime'].fillna(df['closed_datetime']).fillna(df['modified_datetime'])

df['resolution_min_raw'] = (df['end_time'] - df['start_datetime']).dt.total_seconds() / 60

neg_mask = df['resolution_min_raw'] < 0
print(f'Negative resolution times: {neg_mask.sum()}')
df.loc[neg_mask, 'resolution_min_raw'] = np.nan

latest = df['start_datetime'].max()
stale_mask = (
    (df['status'] == 'active') & 
    ((latest - df['start_datetime']).dt.days > 30)
)
print(f'Stale active records (>30 days): {stale_mask.sum()}')
df.loc[stale_mask, 'resolution_min_raw'] = np.nan

valid_mask = df['resolution_min_raw'].notna() & (df['resolution_min_raw'] > 0)
p99 = df.loc[valid_mask, 'resolution_min_raw'].quantile(0.99)
print(f'99th percentile resolution: {p99:.1f} min')
df['resolution_min'] = df['resolution_min_raw'].clip(upper=p99)
df.loc[~valid_mask, 'resolution_min'] = np.nan

df['has_valid_resolution'] = valid_mask & (df['resolution_min_raw'] < p99 * 1.5)

print(f'Records with valid resolution: {df["has_valid_resolution"].sum()} / {len(df)}')
df.loc[df['has_valid_resolution'], 'resolution_min'].describe()

In [ ]:
df['corridor'] = df['corridor'].fillna('Non-corridor').replace('NULL', 'Non-corridor')
print(f'Corridors after cleanup: {df["corridor"].nunique()} unique')

valid_zones = df[(df['zone'].notna()) & (df['zone'] != 'NULL')]
zone_lookup = (
    valid_zones
    .groupby('corridor')['zone']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown')
    .to_dict()
)

def fill_zone(row):
    if pd.notna(row['zone']) and row['zone'] != 'NULL':
        return row['zone']
    return zone_lookup.get(row['corridor'], 'Unknown')

df['zone'] = df.apply(fill_zone, axis=1)
print(f'Zones after backfill: {df["zone"].nunique()} unique')
print(f'Remaining unknown zones: {(df["zone"] == "Unknown").sum()}')

In [ ]:
df['event_cause'] = df['event_cause'].str.lower().str.strip()
df['event_cause'] = df['event_cause'].replace({
    'debris': 'others',
    'test_demo': None,
    'null': 'unknown',
})

df['event_cause'] = df['event_cause'].replace({
    'procession': 'public_event_vip',
    'vip_movement': 'public_event_vip',
    'protest': 'public_event_vip',
    'public_event': 'public_event_vip',
})

before = len(df)
df = df.dropna(subset=['event_cause']).reset_index(drop=True)
print(f'Dropped {before - len(df)} test_demo rows')

df['veh_type'] = df['veh_type'].fillna('unknown').replace('NULL', 'unknown').replace('', 'unknown')
df['veh_type'] = df['veh_type'].str.lower().str.strip()

df['road_closure'] = df['requires_road_closure'].map(
    {True: 1, False: 0, 'TRUE': 1, 'FALSE': 0}
).fillna(0).astype(int)

df['priority_num'] = df['priority'].map({'High': 1, 'Low': 0}).fillna(0).astype(int)

invalid_gps = (df['latitude'].abs() < 1) | (df['longitude'].abs() < 1)
df['valid_gps'] = ~invalid_gps
print(f'Invalid GPS: {invalid_gps.sum()}')

df['is_planned'] = (df['event_type'] == 'planned').astype(int)

df['is_potential_duplicate'] = df.duplicated(
    subset=['latitude', 'longitude', 'event_cause', 'hour'], keep=False
)
print(f'Potential duplicates: {df["is_potential_duplicate"].sum()}')

print(f'\nEvent causes after cleanup:')
df['event_cause'].value_counts()

## 2. Feature Engineering
Temporal features (`hour`/`dayofweek`/`month` plus a single `is_peak_hour` flag for the EDA-verified 19-23 / 4-7 peaks -- tree models don't need cyclic sin/cos encoding). A Kannada-aware NLP pass extracts structural signal (severity/closure/slow keyword hits, description length) from all rows and reclassifies `others`-cause records via keyword matching, falling back to targeted translation only for the records that still have untranslated Kannada text. `reason_breakdown` free text is canonicalized (including common typo variants) into a small set of sub-cause buckets. Remaining steps add ordinal cause severity, a strictly past-only incident density window (no leakage), real osmnx betweenness centrality (zeroed out, not faked, when unavailable), haversine-only HDBSCAN spatial clusters, a congestion index, and a historical per-corridor risk score.

In [ ]:
df['hour'] = df['start_datetime'].dt.hour
df['dayofweek'] = df['start_datetime'].dt.dayofweek
df['month'] = df['start_datetime'].dt.month

df['is_peak_hour'] = (df['hour'].isin(range(19, 24)) | df['hour'].isin(range(4, 8))).astype(int)

print(f'Peak hours (19-23 or 4-7): {df["is_peak_hour"].sum()} records')
print(f'Hour range: {df["hour"].min()}-{df["hour"].max()}, Month range: {df["month"].min()}-{df["month"].max()}')

In [ ]:
KANNADA_RANGE = re.compile(r'[\u0C80-\u0CFF]')

desc_col = 'description' if 'description' in df.columns else 'remarks'
if desc_col not in df.columns:
    print('WARNING: No description column found!')
    desc_col = None
else:
    df[desc_col] = df[desc_col].fillna('')
    print(f'Using column: {desc_col}')
    print(f'Non-empty descriptions: {(df[desc_col].str.len() > 0).sum()}')

In [ ]:
def extract_nlp_features(text):
    """Extract features from raw Unicode text - no translation needed."""
    text_lower = str(text).lower()
    
    severity_kw = ['stuck', 'blocked', 'closed', 'jam', 'accident', 'fire',
                   '\u0CA8\u0CBF\u0C82\u0CA4\u0CBF\u0CA6\u0CC6', '\u0CA8\u0CBF\u0CB2\u0CCD\u0CB2\u0CBF\u0CB8\u0CBF', '\u0CAE\u0CC1\u0CB0\u0CBF\u0CA6']
    slow_kw = ['slow', 'congestion', 'heavy', '\u0CA8\u0CBF\u0CA7\u0CBE\u0CA8', '\u0CB8\u0CCD\u0CB2\u0CCB\u0CAE\u0CC2\u0CAE\u0CC6\u0C82\u0C9F\u0CCD']
    no_impact_kw = ['no problem', 'normal', 'clear', 'ok', 'fine', '\u0CB8\u0CB0\u0CBF', '\u0CA8\u0CBE\u0CB0\u0CCD\u0CAE\u0CB2\u0CCD']
    closure_kw = ['closed', 'blocked', 'one side', '\u0CAE\u0CC1\u0C9A\u0CCD\u0C9A\u0CBF', '\u0CAC\u0CCD\u0CB2\u0CBE\u0C95\u0CCD']
    
    severity = sum(1 for kw in severity_kw if kw in text_lower)
    slow = int(any(kw in text_lower for kw in slow_kw))
    no_impact = int(any(kw in text_lower for kw in no_impact_kw))
    closure = int(any(kw in text_lower for kw in closure_kw))
    has_kannada = int(bool(KANNADA_RANGE.search(text)))
    
    return {
        'nlp_severity_score': min(severity, 5),
        'nlp_mentions_closure': closure,
        'nlp_mentions_slow': slow,
        'nlp_no_impact': no_impact,
        'description_length': len(text),
        'has_kannada': has_kannada,
    }

if desc_col:
    print('Extracting NLP features from raw text...')
    nlp_features = df[desc_col].apply(extract_nlp_features).apply(pd.Series)
    for col in nlp_features.columns:
        df[col] = nlp_features[col]
    
    print(f'Records with Kannada text: {df["has_kannada"].sum()}')
    print(f'Records with severity > 0: {(df["nlp_severity_score"] > 0).sum()}')
    print(f'Records mentioning closure: {df["nlp_mentions_closure"].sum()}')
    print(f'Records mentioning slow: {df["nlp_mentions_slow"].sum()}')
    print(f'Records with no impact: {df["nlp_no_impact"].sum()}')
else:
    for c in ['nlp_severity_score','nlp_mentions_closure','nlp_mentions_slow','nlp_no_impact','description_length','has_kannada']:
        df[c] = 0

In [ ]:
df['reclassified_cause'] = df['event_cause'].copy()

others_mask = df['event_cause'] == 'others'
print(f"'others' records to reclassify: {others_mask.sum()}")

CAUSE_KEYWORDS = {
    'vehicle_breakdown': ['breakdown', 'puncture', 'tyre', 'brake', 'engine', 'broke',
                          'off road', 'offroad', 'off-road', 'gear', 'clutch', 'diesel',
                          '\u0C95\u0CC6\u0C9F\u0CCD\u0C9F\u0CC1', '\u0CAA\u0C82\u0C9A\u0CB0\u0CCD', '\u0CAC\u0CCD\u0CB0\u0CC7\u0C95\u0CCD'],
    'accident': ['accident', 'collision', 'hit', 'crash', '\u0C85\u0CAA\u0C98\u0CBE\u0CA4'],
    'pot_holes': ['pothole', 'pit', 'crater', 'manhole', 'chamber', '\u0CB9\u0CCA\u0C82\u0CA1'],
    'water_logging': ['water', 'flood', 'waterlog', 'rain', '\u0CA8\u0CC0\u0CB0\u0CC1', '\u0CAE\u0CB3\u0CC6'],
    'tree_fall': ['tree', 'branch', 'fell', '\u0CAE\u0CB0'],
    'construction': ['construction', 'road work', 'digging', '\u0CA8\u0CBF\u0CB0\u0CCD\u0CAE\u0CBE\u0CA3'],
    'congestion': ['traffic jam', 'congestion', 'heavy traffic', 'signal off'],
}

reclassified_count = 0
if desc_col:
    for idx in df[others_mask].index:
        text = str(df.at[idx, desc_col]).lower()
        for cause, keywords in CAUSE_KEYWORDS.items():
            if any(kw in text for kw in keywords):
                df.at[idx, 'reclassified_cause'] = cause
                reclassified_count += 1
                break

print(f'Reclassified from others (keywords): {reclassified_count}')

In [ ]:
remaining_others = (df['reclassified_cause'] == 'others') & (df.get('has_kannada', pd.Series([0]*len(df))) == 1)
print(f'Remaining others with Kannada text (attempting translation): {remaining_others.sum()}')

try:
    from deep_translator import GoogleTranslator
    translator = GoogleTranslator(source='kn', target='en')
    
    TRANSLATE_LIMIT = 300
    translate_idx = df[remaining_others].index[:TRANSLATE_LIMIT]
    translated_count = 0
    
    for idx in translate_idx:
        try:
            text = str(df.at[idx, desc_col])
            if len(text.strip()) < 3:
                continue
            translated = translator.translate(text)
            if translated:
                translated_lower = translated.lower()
                for cause, keywords in CAUSE_KEYWORDS.items():
                    if any(kw in translated_lower for kw in keywords):
                        df.at[idx, 'reclassified_cause'] = cause
                        translated_count += 1
                        break
            time.sleep(0.5)
        except Exception:
            pass
    
    print(f'Reclassified via translation: {translated_count}')
except ImportError:
    print('deep-translator not installed -- skipping translation step')

print(f'\nFinal reclassified cause distribution:')
df['reclassified_cause'].value_counts()

In [ ]:
KANNADA_RANGE_RB = re.compile(r'[\u0C80-\u0CFF]')

BREAKDOWN_REASON_KEYWORDS = {
    'tyre_puncture': ['punct', 'puncher', 'punches'],
    'tyre_burst': ['burst', 'brust', 'blast', 'burest'],
    'diesel_empty': ['diesel', 'desel', 'disel', 'desail', 'fuel'],
    'engine_problem': ['engine', 'engin', 'engeen', 'radiator'],
    'battery_problem': ['battery', 'battiry'],
    'electrical_problem': ['wiring', 'wire burn', 'electricity'],
    'axle_problem': ['axel', 'axile', 'axle'],
    'vacuum_problem': ['vaccum', 'vacum', 'vacuum'],
    'oil_leak': ['oil leak'],
    'gear_problem': ['gear', 'gare'],
    'clutch_problem': ['clutch', 'cluch'],
    'brake_problem': ['brake', 'breck'],
    'bearing_problem': ['bairing', 'bearing'],
    'air_problem': ['air lock', 'air problem', 'aer block'],
    'steering_problem': ['steering'],
    'off_road': ['off road', 'offroad'],
    'starting_problem': ['start'],
    'mechanical_problem': ['mechanical', 'technical'],
    'general_breakdown': ['breakdown', 'break down', 'brek down', 'breaks down',
                          'brekdown', 'breckdown', 'breaddown', 'broke'],
}

def canonicalize_reason(raw):
    if pd.isna(raw) or str(raw).strip() in ('', '.'):
        return 'none'
    text = str(raw)
    if KANNADA_RANGE_RB.search(text):
        try:
            from deep_translator import GoogleTranslator
            text = GoogleTranslator(source='kn', target='en').translate(text) or text
        except Exception:
            pass
    text_norm = re.sub(r'[^a-z0-9\s]', '', text.lower().strip())
    for label, keywords in BREAKDOWN_REASON_KEYWORDS.items():
        if any(kw in text_norm for kw in keywords):
            return label
    return 'other'

df['reason_breakdown_clean'] = df['reason_breakdown'].apply(canonicalize_reason)
print(f"Raw unique values: {df['reason_breakdown'].nunique()}")
print(f"Canonicalized unique values: {df['reason_breakdown_clean'].nunique()}")
df['reason_breakdown_clean'].value_counts()

In [ ]:
reason_counts = df.loc[df['reason_breakdown_clean'] != 'none', 'reason_breakdown_clean'].value_counts()

def normalize_for_display(raw):
    text = str(raw)
    if KANNADA_RANGE_RB.search(text):
        try:
            from deep_translator import GoogleTranslator
            text = GoogleTranslator(source='kn', target='en').translate(text) or text
        except Exception:
            pass
    return re.sub(r'[^a-z0-9\s]', '', text.lower().strip())

other_mask = df['reason_breakdown_clean'] == 'other'
other_examples = df.loc[other_mask, 'reason_breakdown'].dropna().apply(normalize_for_display).value_counts()
top_other = other_examples.head(8)
legend_text = 'Top items inside "other" (n={}):\n'.format(other_mask.sum()) + '\n'.join(
    f'- {label} ({count})' for label, count in top_other.items()
)

fig, ax = plt.subplots(figsize=(13, 7))
ax.barh(reason_counts.index[::-1], reason_counts.values[::-1], color='#4A90D9', edgecolor='white')
ax.set_xlabel('Count', fontsize=12)
ax.set_title('Vehicle Breakdown Sub-Cause (Canonicalized)', fontsize=13, fontweight='bold')
for i, v in enumerate(reason_counts.values[::-1]):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
ax.text(
    1.02, 0.5, legend_text, transform=ax.transAxes, fontsize=8.5, va='center', ha='left',
    bbox=dict(boxstyle='round', facecolor='#F5F5F5', edgecolor='#999999')
)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'breakdown_cause_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
CAUSE_SEVERITY = {
    'accident': 5, 'tree_fall': 4, 'water_logging': 4,
    'road_conditions': 3, 'pot_holes': 3, 'public_event_vip': 3, 'fog / low visibility': 3,
    'congestion': 2, 'vehicle_breakdown': 2, 'construction': 2,
    'others': 2, 'unknown': 1,
}

df['cause_severity'] = df['reclassified_cause'].map(CAUSE_SEVERITY).fillna(1).astype(int)
df['cause_severity'].value_counts()

In [ ]:
df = df.sort_values('start_datetime').reset_index(drop=True)

print('Computing leakage-free incident density (takes ~2-3 min)...')
start_t = time.time()

density_1h = []
density_24h = []

for i, row in df.iterrows():
    if i % 2000 == 0:
        print(f'  Processing row {i}/{len(df)}...')
    
    if i == 0:
        density_1h.append(0)
        density_24h.append(0)
        continue
    
    same_corridor = df.loc[:i-1, 'corridor'] == row['corridor']
    past_df = df.loc[:i-1][same_corridor]
    
    if len(past_df) == 0:
        density_1h.append(0)
        density_24h.append(0)
        continue
    
    time_diff = (row['start_datetime'] - past_df['start_datetime']).dt.total_seconds() / 3600
    density_1h.append((time_diff <= 1).sum())
    density_24h.append((time_diff <= 24).sum())

df['incident_density_1h'] = density_1h
df['incident_density_24h'] = density_24h

elapsed = time.time() - start_t
print(f'Done in {elapsed:.1f}s')
print(f'Density 1h stats: mean={np.mean(density_1h):.2f}, max={max(density_1h)}')
print(f'Density 24h stats: mean={np.mean(density_24h):.2f}, max={max(density_24h)}')

In [ ]:
ATTEMPT_OSMNX = False
centrality_is_real = False
try:
    if not ATTEMPT_OSMNX:
        raise ImportError('osmnx attempt disabled (ATTEMPT_OSMNX=False)')
    import osmnx as ox
    import networkx as nx
    import pickle
    
    GRAPH_PATH = 'data/bengaluru_graph.graphml'
    BC_PATH = 'data/betweenness.pkl'
    
    if os.path.exists(GRAPH_PATH):
        print('Loading cached Bengaluru graph...')
        G = ox.load_graphml(GRAPH_PATH)
    else:
        print('Downloading Bengaluru drive network (~3 min)...')
        G = ox.graph_from_place('Bengaluru, Karnataka, India', network_type='drive')
        ox.save_graphml(G, GRAPH_PATH)
        print(f'Graph saved: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    
    if os.path.exists(BC_PATH):
        print('Loading cached betweenness centrality...')
        with open(BC_PATH, 'rb') as f:
            bc = pickle.load(f)
    else:
        print('Computing betweenness centrality (~5 min)...')
        bc = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()), weight='length')
        with open(BC_PATH, 'wb') as f:
            pickle.dump(bc, f)
        print(f'Centrality computed for {len(bc)} nodes')
    
    def get_centrality(lat, lon):
        try:
            node = ox.distance.nearest_nodes(G, lon, lat)
            return bc.get(node, 0.0)
        except:
            return 0.0
    
    print('Mapping incidents to road network nodes...')
    valid = df[df['valid_gps']].copy()
    df['node_centrality'] = 0.0
    df.loc[valid.index, 'node_centrality'] = valid.apply(
        lambda r: get_centrality(r['latitude'], r['longitude']), axis=1
    )
    centrality_is_real = True
    
except ImportError:
    print('osmnx not installed -- centrality signal unavailable (NOT faked with a frequency proxy)')
    df['node_centrality'] = 0.0
except Exception as e:
    print(f'osmnx installed but graph download/processing failed ({e!r}) -- centrality signal unavailable')
    df['node_centrality'] = 0.0

df['corridor_centrality_max'] = df.groupby('corridor')['node_centrality'].transform('max')
print(f'centrality_is_real = {centrality_is_real}')
print(f'Centrality stats: mean={df["node_centrality"].mean():.6f}, max={df["node_centrality"].max():.6f}')

In [ ]:
import hdbscan

valid_gps = df[df['valid_gps']].copy()
coords_rad = np.radians(valid_gps[['latitude', 'longitude']].values)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric='haversine',
    cluster_selection_method='eom'
)

labels = clusterer.fit_predict(coords_rad)

df['hotspot_id'] = -1
df.loc[valid_gps.index, 'hotspot_id'] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f'Clusters found: {n_clusters}')
print(f'Noise points: {n_noise}')
print(f'Clustered points: {(labels >= 0).sum()}')

cluster_temporal = (
    df[df['hotspot_id'] >= 0]
    .groupby('hotspot_id')['hour']
    .agg(['mean', 'std', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else -1])
)
cluster_temporal.columns = ['mean_hour', 'std_hour', 'peak_hour']

joblib.dump(clusterer, os.path.join(MODELS_DIR, 'hdbscan_clusters.joblib'))
print('\nTop 10 cluster temporal profiles:')
cluster_temporal.head(10)

In [ ]:
corridor_hour = (
    df.groupby(['corridor', 'hour'])
    .agg(
        incident_count=('event_cause', 'size'),
        avg_resolution=('resolution_min', 'mean'),
        closure_count=('road_closure', 'sum'),
        severity_sum=('cause_severity', 'sum'),
    )
    .reset_index()
)

corridor_hour['closure_rate'] = (
    corridor_hour['closure_count'] / corridor_hour['incident_count']
).fillna(0)

max_count = corridor_hour['incident_count'].max()
corridor_hour['congestion_index'] = (
    (corridor_hour['incident_count'] / max_count) * 
    (1 + 0.5 * corridor_hour['closure_rate'])
)

df = df.merge(
    corridor_hour[['corridor', 'hour', 'congestion_index']],
    on=['corridor', 'hour'], how='left'
)
df['congestion_index'] = df['congestion_index'].fillna(0)

print('Top 10 corridor-hour combos by congestion:')
corridor_hour.nlargest(10, 'congestion_index')[['corridor', 'hour', 'incident_count', 'congestion_index']]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

corridor_stats = df.groupby('corridor').agg(
    incident_count=('event_cause', 'size'),
    avg_resolution=('resolution_min', 'mean'),
    closure_rate=('road_closure', 'mean'),
    max_centrality=('corridor_centrality_max', 'max'),
    avg_congestion_idx=('congestion_index', 'mean'),
    n_hotspot_incidents=('hotspot_id', lambda x: (x >= 0).sum()),
).reset_index().fillna(0)

components = ['incident_count', 'avg_resolution', 'closure_rate',
              'max_centrality', 'avg_congestion_idx', 'n_hotspot_incidents']
scaler = MinMaxScaler()
corridor_stats[components] = scaler.fit_transform(corridor_stats[components])

if centrality_is_real:
    weights = dict(incident_count=0.28, avg_resolution=0.22, max_centrality=0.20,
                   closure_rate=0.15, avg_congestion_idx=0.10, n_hotspot_incidents=0.05)
else:
    weights = dict(incident_count=0.32, avg_resolution=0.26, max_centrality=0.0,
                   closure_rate=0.21, avg_congestion_idx=0.14, n_hotspot_incidents=0.07)

corridor_stats['risk_score'] = sum(weights[c] * corridor_stats[c] for c in components) * 100
corridor_stats = corridor_stats.sort_values('risk_score', ascending=False)

corridor_stats.to_csv(os.path.join(PROCESSED_DIR, 'corridor_risk_scores.csv'), index=False)
joblib.dump(corridor_stats, os.path.join(MODELS_DIR, 'corridor_risk_scores.joblib'))

print(f'Weights used (centrality_is_real={centrality_is_real}): {weights}')
print('\nCorridor Risk Scores, named corridors only (top 15):')
corridor_stats[corridor_stats['corridor'] != 'Non-corridor'][['corridor', 'risk_score']].head(15)

## 3. EDA Plots
Hourly distribution, corridor risk leaderboard, cause breakdown, resolution-time distribution, HDBSCAN cluster map, day-of-week pattern.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
hourly = df['hour'].value_counts().sort_index()
colors = ['#D85A30' if h in range(19, 24) or h in range(4, 8) 
          else '#4A90D9' if h in range(10, 17) 
          else '#888888' for h in range(24)]
ax.bar(hourly.index, hourly.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Incident Count', fontsize=12)
ax.set_title('Incident Distribution by Hour -- Actual Peaks Highlighted', fontsize=14, fontweight='bold')
ax.set_xticks(range(24))
ax.legend(handles=[
    Patch(color='#D85A30', label='Peak Hours (19-23, 4-7)'),
    Patch(color='#4A90D9', label='Quiet Hours (10-16)'),
    Patch(color='#888888', label='Other Hours'),
], loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'hourly_dist_annotated.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
top15 = corridor_stats[corridor_stats['corridor'] != 'Non-corridor'].head(15).sort_values('risk_score')
colors_bar = ['#8B0000' if s > 70 else '#D85A30' if s > 50 else '#1D9E75' for s in top15['risk_score']]
ax.barh(top15['corridor'], top15['risk_score'], color=colors_bar, edgecolor='white')
ax.set_xlabel('Risk Score (0-100)', fontsize=12)
ax.set_title('Top 15 Named Corridors by Historical Risk Score', fontsize=14, fontweight='bold')
for i, (v, c) in enumerate(zip(top15['risk_score'], top15['corridor'])):
    ax.text(v + 1, i, f'{v:.1f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'corridor_risk.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cause_counts = df['reclassified_cause'].value_counts()
ax.bar(range(len(cause_counts)), cause_counts.values, color='#4A90D9', edgecolor='white')
ax.set_xticks(range(len(cause_counts)))
ax.set_xticklabels(cause_counts.index, rotation=45, ha='right')
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Incident Causes (After NLP Reclassification)', fontsize=14, fontweight='bold')
for i, v in enumerate(cause_counts.values):
    ax.text(i, v + 20, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'cause_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
valid_res = df[df['has_valid_resolution']]['resolution_min']
ax.hist(valid_res, bins=50, color='#2E8B57', edgecolor='white', alpha=0.8)
ax.axvline(valid_res.median(), color='red', linestyle='--', linewidth=2, label=f'Median: {valid_res.median():.0f} min')
ax.axvline(valid_res.mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean: {valid_res.mean():.0f} min')
ax.set_xlabel('Resolution Time (minutes)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Resolution Time Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'resolution_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
noise = df[(df['hotspot_id'] == -1) & df['valid_gps']]
clustd = df[(df['hotspot_id'] >= 0) & df['valid_gps']]
ax.scatter(noise['longitude'], noise['latitude'], c='lightgray', s=3, alpha=0.3, label='Noise')
if len(clustd) > 0:
    scatter = ax.scatter(clustd['longitude'], clustd['latitude'], 
                       c=clustd['hotspot_id'], cmap='tab20', s=10, alpha=0.7)
    plt.colorbar(scatter, ax=ax, label='Cluster ID')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('HDBSCAN Incident Hotspot Clusters -- Bengaluru', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'hdbscan_clusters.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
dow_counts = df['dayofweek'].value_counts().sort_index()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax.bar(range(7), [dow_counts.get(i, 0) for i in range(7)], 
       color=['#D85A30' if i >= 5 else '#4A90D9' for i in range(7)], edgecolor='white')
ax.set_xticks(range(7))
ax.set_xticklabels(days)
ax.set_ylabel('Incident Count', fontsize=12)
ax.set_title('Incidents by Day of Week', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'dow_pattern.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Selection & Save
Correlation + mutual-information against both targets (`resolution_min`, `reclassified_cause`) decides which engineered columns actually carry signal; everything else is dropped before saving the pruned, model-ready dataset and lookup tables.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

candidate_features = [
    'hour', 'dayofweek', 'month', 'is_peak_hour',
    'node_centrality', 'corridor_centrality_max', 'hotspot_id',
    'congestion_index', 'road_closure', 'priority_num', 'cause_severity', 'is_planned',
    'incident_density_1h', 'incident_density_24h',
    'nlp_severity_score', 'nlp_mentions_closure', 'nlp_mentions_slow', 'nlp_no_impact',
    'description_length', 'has_kannada',
]
candidate_features = [c for c in candidate_features if c in df.columns]

corr_matrix = df[candidate_features].corr()
print('=== Redundant feature pairs (|r| > 0.5) ===')
redundant_pairs = []
for i, a in enumerate(candidate_features):
    for b in candidate_features[i+1:]:
        r = corr_matrix.loc[a, b]
        if abs(r) > 0.5:
            redundant_pairs.append((a, b, r))
for a, b, r in sorted(redundant_pairs, key=lambda x: -abs(x[2])):
    print(f'  {a:25s} {b:25s} r={r:.3f}')

valid = df[df['has_valid_resolution']]
print('\n=== Correlation with resolution_min ===')
res_corr = {c: valid[c].corr(valid['resolution_min']) for c in candidate_features}
for c, r in sorted(res_corr.items(), key=lambda x: -abs(x[1])):
    print(f'  {c:25s} r={r:.4f}')

X = df[candidate_features].fillna(0)
y = LabelEncoder().fit_transform(df['reclassified_cause'])
mi_scores = mutual_info_classif(X, y, discrete_features='auto', random_state=42)
print('\n=== Mutual information with reclassified_cause ===')
for c, m in sorted(zip(candidate_features, mi_scores), key=lambda x: -x[1]):
    print(f'  {c:25s} mi={m:.4f}')

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(candidate_features)))
ax.set_yticks(range(len(candidate_features)))
ax.set_xticklabels(candidate_features, rotation=90, fontsize=8)
ax.set_yticklabels(candidate_features, fontsize=8)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Engineered Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'feature_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ID_AND_TARGET_COLS = [
    'start_datetime', 'corridor', 'zone', 'latitude', 'longitude',
    'reclassified_cause', 'resolution_min', 'has_valid_resolution',
    'event_type', 'status',
]

ENGINEERED_COLS = [
    'hour', 'dayofweek', 'month', 'is_peak_hour',
    'hotspot_id', 'congestion_index', 'cause_severity', 'road_closure', 'is_planned',
    'incident_density_24h', 'nlp_severity_score', 'description_length', 'has_kannada',
    'veh_type', 'reason_breakdown_clean',
]
if centrality_is_real:
    ENGINEERED_COLS.append('corridor_centrality_max')

ML_COLS = ID_AND_TARGET_COLS + ENGINEERED_COLS

ml_cols_present = [c for c in ML_COLS if c in df.columns]
missing = [c for c in ML_COLS if c not in df.columns]
if missing:
    print(f'Missing columns (skipped): {missing}')

df_out = df[ml_cols_present].copy()

for col in df_out.select_dtypes(include=['datetimetz']).columns:
    df_out[col] = df_out[col].dt.tz_localize(None)

output_path = os.path.join(PROCESSED_DIR, 'clean_featured.parquet')
df_out.to_parquet(output_path, index=False)

csv_path = os.path.join(PROCESSED_DIR, 'clean_featured.csv')
df_out.to_csv(csv_path, index=False)

verify = pd.read_parquet(output_path)
print(f'Saved: {output_path}')
print(f'Shape: {verify.shape}  ({len(ID_AND_TARGET_COLS)} id/target cols + {len(ENGINEERED_COLS)} engineered features)')
print(f'\nColumn types:')
verify.dtypes

## Day 1 Complete - Deliverables

| Deliverable | Description |
|---|---|
| `data/processed/clean_featured.parquet` | Full featured dataset, pruned feature set |
| `data/processed/corridor_risk_scores.csv` | Risk scores per corridor |
| `models/trained/hdbscan_clusters.joblib` | Spatial cluster model |
| `models/trained/corridor_risk_scores.joblib` | Risk score table |
| `plots/*.png` | EDA plots (hourly, corridor risk, cause breakdown, resolution dist, HDBSCAN, day-of-week) |

### Day 2 Handoff
Load `data/processed/clean_featured.parquet` and train:
- **CatBoost regressor** -> `resolution_min` (may use `cause_severity`)
- **CatBoost classifier** -> `reclassified_cause` with `auto_class_weights='Balanced'` (must **drop** `cause_severity` from its inputs -- it's derived from the target)